# CIFAR-10 with ResNet-20 Baseline
This notebook downloads the CIFAR-10 dataset, clones the project repository to access the model definition, and trains a ResNet-20 model.

In [ ]:
!git clone https://github.com/MrRoboto11102003/Quantization_project.git
import sys
sys.path.append('Quantization_project')

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from resnet import resnet20

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=100, shuffle=False, num_workers=2)

In [ ]:
net = resnet20().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.1, momentum=0.9, weight_decay=1e-4)
scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[100, 150], gamma=0.1)
epochs = 200

def train(epoch):
    net.train()
    train_loss, correct, total = 0, 0, 0
    for batch_idx, (inputs, targets) in enumerate(trainloader):
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
    print(f'Epoch {epoch}: Train Loss: {train_loss/(batch_idx+1):.3f} | Acc: {100.*correct/total:.3f}%')

def test():
    net.eval()
    test_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(testloader):
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = net(inputs)
            loss = criterion(outputs, targets)
            test_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    print(f'Test Loss: {test_loss/(batch_idx+1):.3f} | Acc: {100.*correct/total:.3f}%')

for epoch in range(epochs):
    train(epoch)
    test()
    scheduler.step()

In [ ]:
!pip install thop
import time
from thop import profile

dummy_input = torch.randn(1, 3, 32, 32).to(device)
flops, params = profile(net, inputs=(dummy_input,), verbose=False)
print(f"FLOPs: {flops / 1e6:.2f} M")
print(f"Params: {params / 1e6:.2f} M")

net.eval()
with torch.no_grad():
    for inputs, _ in testloader:
        inputs = inputs.to(device)
        _ = net(inputs)
    if torch.cuda.is_available():
        torch.cuda.synchronize()

times = []
with torch.no_grad():
    for _ in range(30):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start = time.time()
        
        for inputs, _ in testloader:
            inputs = inputs.to(device)
            _ = net(inputs)
            
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        times.append(time.time() - start)

avg_time = sum(times) / len(times)
print(f"Average Inference Time (30 runs): {avg_time:.4f} seconds")